In [3]:
from tensorflow.keras.models import load_model

model = load_model("model.h5")
print("Model loaded successfully")

Model loaded successfully


In [4]:
import joblib

scaler = joblib.load("scaler.pkl")
print("Scaler loaded")

Scaler loaded


In [6]:
import os
import numpy as np
from scipy.io import loadmat

WINDOW_SIZE = 256
STEP_SIZE = 128

def create_windows(eeg, window_size, step_size):
    windows = []
    for i in range(0, eeg.shape[1] - window_size, step_size):
        windows.append(eeg[:, i:i+window_size])
    return np.array(windows)

def load_folder(folder_path, label):
    X_list = []
    y_list = []

    files = [f for f in os.listdir(folder_path) if f.endswith('.mat')]

    for file in files:
        mat = loadmat(os.path.join(folder_path, file))
        for key in mat.keys():
            if not key.startswith("__"):
                eeg = mat[key]
                break

        eeg = eeg.T
        windows = create_windows(eeg, WINDOW_SIZE, STEP_SIZE)

        X_list.append(windows)
        y_list.extend([label] * len(windows))

    return np.concatenate(X_list), np.array(y_list)

In [7]:
X_adhd1, y_adhd1 = load_folder("data/ADHD_part1", 0)
X_adhd2, y_adhd2 = load_folder("data/ADHD_part2", 0)

X_ctrl1, y_ctrl1 = load_folder("data/Control_part1", 1)
X_ctrl2, y_ctrl2 = load_folder("data/Control_part2", 1)

X = np.concatenate([X_adhd1, X_adhd2, X_ctrl1, X_ctrl2])
y = np.concatenate([y_adhd1, y_adhd2, y_ctrl1, y_ctrl2])

print("Total samples:", X.shape)

Total samples: (16748, 19, 256)


In [8]:
from sklearn.preprocessing import StandardScaler

X_normalized = np.zeros_like(X, dtype=np.float32)

for i in range(X.shape[0]):
    scaler_temp = StandardScaler()
    X_normalized[i] = scaler_temp.fit_transform(X[i].T).T

In [9]:
X_normalized = X_normalized[..., np.newaxis]

print("Shape after reshape:", X_normalized.shape)

Shape after reshape: (16748, 19, 256, 1)


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_normalized, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Test shape:", X_test.shape)


Test shape: (3350, 19, 256, 1)


In [11]:
orig_loss, orig_acc = model.evaluate(X_test, y_test)
print("Original Accuracy:", orig_acc)

105/105 ━━━━━━━━━━━━━━━━━━━━ 2s 10ms/step - accuracy: 0.9015 - loss: 0.7361
Original Accuracy: 0.9014925360679626


In [12]:
CHANNEL_NAMES = [
    "Fp1","Fp2","F7","F3","Fz","F4","F8",
    "T3","C3","Cz","C4","T4",
    "T5","P3","Pz","P4","T6",
    "O1","O2"
]

LEFT_CHANNELS = ["Fp1","F7","F3","T3","C3","T5","P3","O1"]
RIGHT_CHANNELS = ["Fp2","F8","F4","T4","C4","T6","P4","O2"]

LEFT_IDX = [CHANNEL_NAMES.index(ch) for ch in LEFT_CHANNELS]
RIGHT_IDX = [CHANNEL_NAMES.index(ch) for ch in RIGHT_CHANNELS]

In [13]:
X_left_only = X_test.copy()
X_left_only[:, RIGHT_IDX, :, :] = 0

left_loss, left_acc = model.evaluate(X_left_only, y_test)
print("LEFT Hemisphere Only Accuracy:", left_acc)

105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.5907 - loss: 4.4041
LEFT Hemisphere Only Accuracy: 0.590746283531189


In [14]:
X_right_only = X_test.copy()
X_right_only[:, LEFT_IDX, :, :] = 0

right_loss, right_acc = model.evaluate(X_right_only, y_test)
print("RIGHT Hemisphere Only Accuracy:", right_acc)

105/105 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.6391 - loss: 3.2903
RIGHT Hemisphere Only Accuracy: 0.6391044855117798
